# ICDN Architecture Stress Test

Controlled computational stress test of the ICDN architecture across
(n, k) combinations. Modules are instantiated with the best-trial
hyperparameters (`best_trial_params.json`) and random weights, since
forward-pass latency/memory depends on tensor shapes, not weight values.
All results are written to a single CSV (`results/stress_test_icdn.csv`);
the paper panels are just filtered views of that one table.

# Imports and Constants

In [1]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import math

from src.nn.context.context_mlp import SharedProductEncoder
from src.nn.heads.parameter_head import DemandParameterHead
from src.nn.heads.neighbor_selector import SparseNeighborSelector
from src.nn.heads.demand_calculator import DemandCalculator
from src.nn.spline import MultiCubicSplineBasis
from src.multiproduct.context import ProductTokenBuilder, _TIME_COLS, _PROMO_COLS, _PER_PRODUCT_COLS
from src.nn.loss.elasticity_loss import ElasticityLoss

device = "cuda" if torch.cuda.is_available() else "cpu"

with open(Path("../results/best_trial_params.json")) as f:
    best_trial = json.load(f)["params"]

HIDDEN_OPTIONS = {
    "64_32": (64, 32), "128_64": (128, 64), "192_96": (192, 96),
    "256_128": (256, 128), "256_128_64": (256, 128, 64),
}

K_SPLINES  = int(best_trial["N_KNOTS"])
HIDDEN     = HIDDEN_OPTIONS[best_trial["HIDDEN_KEY"]]
BATCH_SIZE = int(best_trial["BATCH_SIZE"])
DROPOUT    = best_trial["DROPOUT"]

ACT        = "gelu"
D_ATTN     = 16
N_STORES, N_BRANDS, N_STYLES = 70, 54, 13
D_STORE = 16
D_BRAND = 8
D_STYLE = 8
N_BASELINE = 5
K_BASELINE = 4

H = HIDDEN[-1]

print(f"K_SPLINES={K_SPLINES}  HIDDEN={HIDDEN}  H={H}  D_ATTN={D_ATTN}")

K_SPLINES=3  HIDDEN=(256, 128, 64)  H=64  D_ATTN=16


# Architecture + shared synthetic-context builder

In [2]:
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

torch.manual_seed(0)

def make_param_head() -> DemandParameterHead:
    return DemandParameterHead(
        hidden_dim=H, K_splines=K_SPLINES, n=N_BASELINE, use_cross=True
    ).eval().to(device)

param_head = make_param_head()
# NOTE on `n=N_BASELINE` above: DemandParameterHead's only trainable weights are
# nn.Linear(hidden_dim -> ...) / nn.Linear(2*hidden_dim -> ...) heads, which
# depend on hidden_dim/K_splines but NOT on n. `n` is only used at __init__ to
# build the *default* `_pairs` index buffer (non-trainable), used when
# run(pairs=None). Every call site in this notebook always passes `pairs=`
# explicitly (built by the neighbor selector for the current n), so the SAME
# weights are reused for any n. Verified at runtime:
assert count_params(DemandParameterHead(hidden_dim=H, K_splines=K_SPLINES, n=5,   use_cross=True)) == \
       count_params(DemandParameterHead(hidden_dim=H, K_splines=K_SPLINES, n=200, use_cross=True)), \
       "DemandParameterHead's parameter count must be n-invariant."

def make_token_builder(n: int) -> ProductTokenBuilder:
    return ProductTokenBuilder(
        n=n,
        n_stores=N_STORES,
        d_store=D_STORE,
        n_brands=N_BRANDS,
        d_brand=D_BRAND,
        n_styles=N_STYLES,
        d_style=D_STYLE,
    ).to(device)
    
_dummy_tb = make_token_builder(N_BASELINE)
D_TOKEN = _dummy_tb.d_token

def make_encoder(training: bool = False) -> SharedProductEncoder:
    module = SharedProductEncoder(d_in=D_TOKEN, hidden=HIDDEN, act=ACT, dropout=DROPOUT).to(device)
    return module.train() if training else module.eval()
    
encoder = make_encoder(training=False)

def make_selector(k_neighbors: int) -> SparseNeighborSelector:
    return SparseNeighborSelector(d_hidden=H, d_attn=D_ATTN, k_neighbors=k_neighbors).eval().to(device)

demand_calc = DemandCalculator()  # shared, stateless — reused by bench_one() and bench_training_step()

def synth_batch(n: int, batch_size: int) -> dict:
    """Synthetic batch matching ProductTokenBuilder.forward()'s expected schema
    (see src/multiproduct/context.py); store/brand/style cardinalities match the
    real dataset, so only the per-sample VALUES are synthetic, not the shapes."""
    return {
        "ids": torch.stack([
            torch.randint(0, N_STORES, (batch_size,), device=device),
            torch.zeros(batch_size, dtype=torch.long, device=device),  # week_id — unused by the builder
        ], dim=1),
        "time_feats":  torch.randn(batch_size, len(_TIME_COLS), device=device),
        "promo_feats": torch.rand(batch_size, len(_PROMO_COLS), device=device),
        "per_prod_float": torch.randn(batch_size, n, len(_PER_PRODUCT_COLS), device=device),
        "per_prod_cat": torch.stack([
            torch.randint(1, N_BRANDS + 1, (batch_size, n), device=device),  # brand_i
            torch.randint(1, N_STYLES + 1, (batch_size, n), device=device),  # style_i
        ], dim=-1),
    }

def synth_meta(n: int, n_categories: int = 3):
    """Synthetic category/brand/style/liters, grouped so the top-k selector has
    real 'same category' structure to exploit."""
    category = torch.arange(n, device=device) % n_categories
    brand    = torch.arange(n, device=device) % max(2, n_categories)
    style    = torch.arange(n, device=device) % max(2, n_categories)
    liters   = torch.empty(n, device=device).uniform_(0.3, 2.0)
    return category, brand, style, liters

def synth_splines(n: int, K: int) -> MultiCubicSplineBasis:
    q = torch.linspace(0.05, 0.95, K)
    base_knots = torch.distributions.Normal(0, 1).icdf(q)
    knots = base_knots.unsqueeze(0).repeat(n, 1)
    return MultiCubicSplineBasis(knots=knots, shift=torch.zeros(n), scale=torch.ones(n)).to(device)

def build_synthetic_context(n: int, k: int):
    """All (n, k)-dependent synthetic assets shared by every benchmark function."""
    k_eff = min(k, n - 1)
    category, brand, style, liters = synth_meta(n)
    splines = synth_splines(n, K_SPLINES)
    selector = make_selector(k_eff)
    return k_eff, category, brand, style, liters, splines, selector

n_params = {
    "encoder":           count_params(encoder),
    "param_head":        count_params(param_head),
    "neighbor_selector": count_params(make_selector(K_BASELINE)),
    "token_builder":     count_params(_dummy_tb),  # synthetic sizes — see note above
}
core_shared_trainable_params = sum(n_params.values())
print("Core shared trainable parameters (n/k-agnostic):", n_params)
print("Core shared trainable parameters (total):", core_shared_trainable_params)
print("NOTE: matches the real ICDN's trainable parameters (encoder + param_head + "
      "neighbor_selector + token_builder's embeddings), except price_splines "
      "(MultiCubicSplineBasis), which has zero trainable parameters — its knots/"
      "shift/scale are non-trainable buffers, not nn.Parameter.")

Core shared trainable parameters (n/k-agnostic): {'encoder': 58560, 'param_head': 2002, 'neighbor_selector': 2051, 'token_builder': 1672}
Core shared trainable parameters (total): 64285
NOTE: matches the real ICDN's trainable parameters (encoder + param_head + neighbor_selector + token_builder's embeddings), except price_splines (MultiCubicSplineBasis), which has zero trainable parameters — its knots/shift/scale are non-trainable buffers, not nn.Parameter.


# GPU Timer

In [3]:
class GpuTimer:
    def __init__(self, device, warmup=3, repeats=15):
        self.device, self.warmup, self.repeats = device, warmup, repeats

    def time_block(self, fn):
        """Runs fn() warmup+repeats times; returns (median_ms, incremental_peak_mb, last_result).

        incremental_peak_mb is the PEAK INCREMENTAL ALLOCATED GPU memory
        attributable to fn() itself: torch.cuda.max_memory_allocated() (since
        the reset below) minus whatever was ALREADY allocated at reset time.
        Without this baseline subtraction, max_memory_allocated() also counts
        tensors that were already live when the counter was reset (e.g. inputs
        built before this block), inflating the reported cost of the block
        under test — this is NOT total peak allocated memory, just the extra
        memory this block allocates on top of what already existed.
        """
        result = None
        for _ in range(self.warmup):
            result = fn()
        baseline_mb = 0.0
        if self.device == "cuda":
            torch.cuda.synchronize()
            baseline_mb = torch.cuda.memory_allocated() / 1e6
            torch.cuda.reset_peak_memory_stats()
        times = []
        for _ in range(self.repeats):
            if self.device == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            result = fn()
            if self.device == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000.0)
        if self.device == "cuda":
            incremental_peak_mb = torch.cuda.max_memory_allocated() / 1e6 - baseline_mb
        else:
            incremental_peak_mb = float("nan")
        return float(np.median(times)), incremental_peak_mb, result

timer = GpuTimer(device)

# Bench One

In [4]:
def compute_selected_elasticities(beta_cross, w_cross, u, Bx_i, dBx_j, attn):
    """Cross-price elasticity E_ij on the selected edges only — mirrors the
    off-diagonal formula inside DemandCalculator.run(return_E=True), exposed
    standalone so both the granular and the end-to-end blocks below can reuse
    the exact same computation instead of duplicating it."""
    return (
        beta_cross
        + (w_cross * dBx_j).sum(dim=-1)
        + torch.einsum("bpk,bpkl,bpl->bp", Bx_i, u, dBx_j)
    ) * attn

def materialize_dense_E(eps_hat, E_cross, i_idx, j_idx, n, device):
    """Scatters {E_ii} (own) and {E_ij} (selected cross) into a dense (B,n,n) matrix."""
    B = eps_hat.shape[0]
    E = torch.zeros(B, n, n, device=device)
    diag = torch.arange(n, device=device)
    E[:, diag, diag] = eps_hat
    E[:, i_idx, j_idx] = E_cross
    return E

@torch.no_grad()
def bench_one(n: int, k: int, batch_size: int = BATCH_SIZE):
    k_eff, category, brand, style, liters, splines, selector = build_synthetic_context(n, k)
    token_builder = make_token_builder(n)
    batch = synth_batch(n, batch_size)
    x = torch.randn(batch_size, n, device=device) * 0.1

    # ── Token construction: store/brand/style embedding lookups + concat, O(n) ──
    tokens_ms, mem_tok, tokens = timer.time_block(lambda: token_builder(batch))

    # ── Shared encoder forward: per-product MLP, O(n) ──
    encoder_ms, mem_enc, h = timer.time_block(lambda: encoder(tokens))

    # ── Spline basis + derivatives, O(n) ──
    spline_ms, mem_spl, (Bx, dBx, _) = timer.time_block(lambda: splines(x))

    # ── Candidate scoring: dense (n,n) score matrix (q·k + meta_bonus) ──
    h_score_sample = encoder(token_builder(synth_batch(n, 256)))
    scoring_ms, mem1, mean_scores = timer.time_block(
        lambda: selector.accumulate_mean_scores([h_score_sample], category, brand, style, liters)
    )

    # ── Top-k: edge selection from the score matrix ──
    not_self, same_cat, meta_bonus = selector._meta_bonus(category, brand, style, liters, device)
    topk_ms, mem2, pairs = timer.time_block(
        lambda: selector._build_pairs(mean_scores, not_self, same_cat)
    )
    i_idx, j_idx = pairs[0], pairs[1]

    # Freeze the graph: from here on, cost is O(n*k_eff), not O(n^2).
    selector.frozen_pairs = pairs
    selector.frozen_edge_bonus = meta_bonus[i_idx, j_idx]

    # ── Parameter/attention + own-price terms, O(n) — granular breakdown ──
    def own_terms():
        _, attn = selector.run(h, category, brand, style, liters)
        params = param_head.run(h, pairs=pairs)
        b, beta, w = params["b"], params["beta"], params["w"]
        y_hat   = b + beta * x + (w * Bx).sum(dim=-1)
        eps_hat = beta + (w * dBx).sum(dim=-1)
        return params, attn, eps_hat
    demand_ms, mem3, (params, attn, eps_hat) = timer.time_block(own_terms)

    # ── Selected cross-interaction terms (feeds y_hat/eps_hat), O(n*k_eff) ──
    Bx_i, Bx_j   = Bx[:, i_idx, :],  Bx[:, j_idx, :]
    dBx_i, dBx_j = dBx[:, i_idx, :], dBx[:, j_idx, :]
    x_j = x[:, j_idx]

    def interaction_eval():
        contrib_yi = (
            params["beta_cross"] * x_j
            + (params["w_cross"] * Bx_j).sum(dim=-1)
            + torch.einsum('bpk,bpkl,bpl->bp', Bx_i, params["u"], Bx_j)
        ) * attn
        contrib_ei = torch.einsum('bpk,bpkl,bpl->bp', dBx_i, params["u"], Bx_j) * attn
        return contrib_yi, contrib_ei
    interaction_ms, mem4, _ = timer.time_block(interaction_eval)

    # ── Selected-edge elasticity computation, O(n*k_eff) — granular breakdown ──
    def selected_elasticities():
        return compute_selected_elasticities(
            params["beta_cross"], params["w_cross"], params["u"], Bx_i, dBx_j, attn
        )
    selected_elast_ms, mem5, E_cross = timer.time_block(selected_elasticities)

    # ── Optional dense materialization, O(n^2) — granular breakdown ──
    dense_ms, mem6, E_dense = timer.time_block(
        lambda: materialize_dense_E(eps_hat, E_cross, i_idx, j_idx, n, device)
    )

    # ── True END-TO-END frozen-graph evaluation, each measured as a SINGLE
    # timed block (not a sum of medians of the sub-blocks above). This is
    # exactly what an inference call pays, from tokens to elasticities. ──
    def frozen_eval_sparse():
        tokens_e       = token_builder(batch)
        h_e            = encoder(tokens_e)
        Bx_e, dBx_e, _ = splines(x)
        _, attn_e      = selector.run(h_e, category, brand, style, liters)
        params_e       = param_head.run(h_e, pairs=pairs)
        _, eps_hat_e, _ = demand_calc.run(
            b=params_e["b"], beta=params_e["beta"], w=params_e["w"],
            x=x, Bx=Bx_e, dBx=dBx_e,
            beta_cross=params_e["beta_cross"], w_cross=params_e["w_cross"], u=params_e["u"],
            pairs=pairs, attn_weights=attn_e, return_E=False,
        )
        E_cross_e = compute_selected_elasticities(
            params_e["beta_cross"], params_e["w_cross"], params_e["u"],
            Bx_e[:, i_idx, :], dBx_e[:, j_idx, :], attn_e,
        )
        return eps_hat_e, E_cross_e

    def frozen_eval_dense():
        eps_hat_e, E_cross_e = frozen_eval_sparse()
        return materialize_dense_E(eps_hat_e, E_cross_e, i_idx, j_idx, n, device)

    e2e_sparse_ms, mem_e2e_sparse, (eps_hat_s, E_cross_s) = timer.time_block(frozen_eval_sparse)
    e2e_dense_ms,  mem_e2e_dense,  E_dense_e2e             = timer.time_block(frozen_eval_dense)

    # ── Output memory: the FULL sparse elasticity output is {E_ii} (own) plus
    # {E_ij} (selected cross); indices are batch-shared, reported separately. ──
    sparse_values_mb = (
        eps_hat_s.element_size() * eps_hat_s.nelement()
        + E_cross_s.element_size() * E_cross_s.nelement()
    ) / 1e6
    dense_values_mb   = E_dense_e2e.element_size() * E_dense_e2e.nelement() / 1e6
    sparse_indices_mb = (
        i_idx.element_size() * i_idx.nelement() + j_idx.element_size() * j_idx.nelement()
    ) / 1e6

    return {
        "n": n, "k": k, "k_eff": k_eff,
        "Graph density": round(k_eff / (n - 1), 4) if n > 1 else float("nan"),
        "Token construction ms": round(tokens_ms, 4),
        "Encoder forward ms": round(encoder_ms, 4),
        "Spline basis and derivatives ms": round(spline_ms, 4),
        "Candidate scoring per batch (ms)": round(scoring_ms, 4),
        "Top-k ms": round(topk_ms, 4),
        "Parameter/attention + own-term ms": round(demand_ms, 4),
        "Selected cross-interaction ms": round(interaction_ms, 4),
        "Selected-edge elasticity computation ms": round(selected_elast_ms, 4),
        "Dense E materialization ms": round(dense_ms, 4),
        "End-to-end frozen-graph evaluation ms (sparse)": round(e2e_sparse_ms, 4),
        "End-to-end frozen-graph evaluation ms (+ dense E)": round(e2e_dense_ms, 4),
        "Sparse elasticity values memory (MB)": round(sparse_values_mb, 6),
        "Dense elasticity values memory (MB)": round(dense_values_mb, 6),
        "Sparse graph indices memory (MB, batch-shared, one-off)": round(sparse_indices_mb, 6),
        "Incremental peak post-encoder graph-selection GPU MB":                round(max(mem1, mem2), 2),
        "Peak end-to-end sparse evaluation GPU MB":    round(mem_e2e_sparse, 2),
        "Peak end-to-end dense-E evaluation GPU MB":   round(mem_e2e_dense, 2),
    }

# Bench Training Step

In [ ]:
elasticity_loss = ElasticityLoss(
    huber_delta=1.0, lambda_smooth=float(best_trial["LAMBDA_SMOOTH"]), lambda_elast=float(best_trial["LAMBDA_ELAST"]),
).to(device)

def bench_training_step(n: int, k: int, batch_size: int = BATCH_SIZE):
    # frozen_pairs stays None on purpose: real training never freezes the graph —
    # it always recomputes candidate scoring + top-k every batch; freeze_graph()
    # is called only ONCE, after training, before eval.
    _, category, brand, style, liters, splines, selector = build_synthetic_context(n, k)
    token_builder = make_token_builder(n)
    # Fresh encoder/param_head per (n, k) combination. encoder/param_head are
    # mutated by optimizer.step() during this benchmark; reusing the module-level
    # globals (shared with bench_one/n_params) across grid points would let
    # gradient updates — and any NaN/Inf blow-up, missed GradScaler overflow
    # detection, or skipped updates — leak from one (n, k) into the next,
    # contaminating comparability across the grid. token_builder/selector above
    # are already rebuilt per call; encoder/param_head now are too.
    encoder_local    = make_encoder(training=True)
    param_head_local = make_param_head()
    trainable = (
        list(token_builder.parameters())
        + list(encoder_local.parameters())
        + list(param_head_local.parameters())
        + list(selector.parameters())
    )
    optimizer = torch.optim.AdamW(trainable, lr=float(best_trial["LR_P1"]))
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None
    batch    = synth_batch(n, batch_size)
    x        = torch.randn(batch_size, n, device=device) * 0.1
    y_target = torch.randn(batch_size, n, device=device)
    obs_mask = torch.ones(batch_size, n, device=device)

    def forward_and_loss():
        tokens        = token_builder(batch)
        h             = encoder_local(tokens)
        Bx, dBx, ddBx = splines(x)
        pairs, attn   = selector.run(h, category, brand, style, liters)
        params        = param_head_local.run(h, pairs=pairs)
        y_hat, eps_hat, E = demand_calc.run(
            b=params["b"], beta=params["beta"], w=params["w"],
            x=x, Bx=Bx, dBx=dBx,
            beta_cross=params["beta_cross"], w_cross=params["w_cross"], u=params["u"],
            pairs=params["pairs"], attn_weights=attn, return_E=True,
        )
        loss, _ = elasticity_loss.run(
            y_hat, y_target, obs_mask,
            w=params["w"], ddBx=ddBx, u=params["u"], Bx=Bx, pairs=params["pairs"], E=E,
        )
        assert torch.isfinite(loss), f"Non-finite loss at n={n}, k={k}: {loss.item()}"
        return loss
        
    def training_step():
        optimizer.zero_grad(set_to_none=True)
        if scaler is not None:
            with torch.amp.autocast("cuda"):
                loss = forward_and_loss()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = forward_and_loss()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
    step_ms, peak_mb, _ = timer.time_block(training_step)
    return step_ms, peak_mb

# Grid and Aggregation

In [6]:
N_OUTER_REPEATS = 15  
n_values = [5, 10, 25, 50, 100, 200]
k_values = [1, 2, 4, 8, 16, 32]

grid = sorted({(n, k) for n in n_values for k in k_values if min(k, n - 1) == k} |
              {(n, n - 1) for n in n_values})  # always include the dense case k=n-1

all_runs = []
for run_idx in range(N_OUTER_REPEATS):
    for n, k in grid:
        row = bench_one(n, k)
        step_ms, peak_train_mb = bench_training_step(n, k)
        row["Training step ms"] = round(step_ms, 4)
        row["Peak training GPU MB"] = round(peak_train_mb, 2)
        row["run_idx"] = run_idx
        all_runs.append(row)
        
df_runs = pd.DataFrame(all_runs)
# Average (and report the spread of) every metric over the N_OUTER_REPEATS
# full sweeps, per (n, k). Averaging AFTER computing raw metrics is correct
# here because every derived column below is a linear combination of these
# metrics, and mean() is linear: mean(a + b) == mean(a) + mean(b).
key_cols    = ["n", "k", "k_eff"]
metric_cols = [c for c in df_runs.columns if c not in key_cols + ["run_idx"]]
df_mean = df_runs.groupby(key_cols, as_index=False)[metric_cols].mean()
df_std  = (
    df_runs.groupby(key_cols, as_index=False)[metric_cols]
    .std()
    .rename(columns={c: f"{c} (std over {N_OUTER_REPEATS} runs)" for c in metric_cols})
)
df_stress = (
    df_mean.merge(df_std, on=key_cols)
    .sort_values(["n", "k"])
    .reset_index(drop=True)
)

# "Online graph-selection time per batch" is what training pays on EVERY
# batch (score matrix + top-k, post-encoder). It is NOT the cost of building
# the FINAL frozen graph: per the paper, the frozen graph is obtained by
# averaging scores over a SINGLE pass over the training split (not every
# epoch), then running top-k ONCE, at the end:
#   T_freeze ≈ N_batches_per_training_split_pass · (T_tokens + T_encoder + T_scoring) + T_top-k
# N_BATCHES_PER_PASS_ESTIMATE below is illustrative. Prefer measuring
# accumulate_mean_scores() + freeze_graph() directly on a real fold instead
# of relying on this estimate.
df_stress["Online graph-selection time per batch (ms)"] = (
    df_stress["Candidate scoring per batch (ms)"] + df_stress["Top-k ms"]
)

N_BATCHES_PER_PASS_ESTIMATE = math.ceil(
    17_795 / BATCH_SIZE
)

# pass over the training split (NOT epochs × batches/epoch)
df_stress[f"Estimated total graph-freeze time, N_batches_per_pass={N_BATCHES_PER_PASS_ESTIMATE} (ms)"] = (
    N_BATCHES_PER_PASS_ESTIMATE * (
        df_stress["Token construction ms"] + df_stress["Encoder forward ms"] + df_stress["Candidate scoring per batch (ms)"]
    ) + df_stress["Top-k ms"]
)

# Sum-of-medians breakdown (useful for attributing cost to individual ops),
# NOT an independently-measured end-to-end number. For the real, single-block
# measurement see "End-to-end frozen-graph evaluation ms" from bench_one.
df_stress["Post-encoder frozen-graph core evaluation ms (sparse)"] = (
    df_stress["Parameter/attention + own-term ms"]
    + df_stress["Selected cross-interaction ms"]
    + df_stress["Selected-edge elasticity computation ms"]
)
df_stress["Post-encoder frozen-graph core evaluation ms (+ dense E)"] = (
    df_stress["Post-encoder frozen-graph core evaluation ms (sparse)"] + df_stress["Dense E materialization ms"]
)

# ── Shared parameter counts: constant across every (n, k) row — see cell 2 ──
df_stress["Core shared trainable params (encoder)"]       = n_params["encoder"]
df_stress["Core shared trainable params (param_head)"]    = n_params["param_head"]
df_stress["Core shared trainable params (selector)"]      = n_params["neighbor_selector"]
df_stress["Core shared trainable params (token_builder)"] = n_params["token_builder"]

OUT_PATH = Path("../results/stress_test_icdn.csv")
df_stress.to_csv(OUT_PATH, index=False)
print(f"Saved to {OUT_PATH}  ({len(df_stress)} rows, {df_stress.shape[1]} columns)")

Saved to ../results/stress_test_icdn.csv  (35 rows, 51 columns)
